In [ ]:
import sys
import importlib.metadata as metadata

required = ["kubernetes", "pandas", "networkx", "dash", "dash-cytoscape", "jupyter-dash"]
versions = {}
missing = []
for package in required:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = None
        missing.append(package)

print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
for package, version in versions.items():
    print(f"{package:20}: {version}")

if missing:
    print("Missing packages:", ", ".join(missing))
    print("Please install them with: pip install dash dash-cytoscape jupyter-dash")
else:
    print("Dependency check passed.")

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import networkx as nx

from kubernetes import client, config
from kubernetes.client import ApiException

# Reuse the config loading logic from notebook 09
def candidate_kubeconfigs() -> list[Path]:
    candidates: list[Path] = []
    env_value = os.getenv("KUBECONFIG")
    if env_value:
        for item in env_value.split(":"):
            path = Path(item).expanduser()
            if path.exists():
                candidates.append(path)
    for path in [Path("~/.kube/config").expanduser(), Path("/etc/rancher/k3s/k3s.yaml")]:
        if path.exists() and path not in candidates:
            candidates.append(path)
    return candidates

def load_k8s_config() -> tuple[str, str | None]:
    errors = []
    for kubeconfig_path in candidate_kubeconfigs():
        try:
            config.load_kube_config(config_file=str(kubeconfig_path))
            return "kubeconfig", str(kubeconfig_path)
        except Exception as exc:
            errors.append(f"{kubeconfig_path}: {exc}")
    try:
        config.load_incluster_config()
        return "in-cluster", None
    except Exception as exc:
        errors.append(f"in-cluster: {exc}")
    raise RuntimeError("Unable to load Kubernetes configuration. Tried: " + " | ".join(errors))

AUTH_MODE, KUBECONFIG_PATH = load_k8s_config()
print("Auth mode       :", AUTH_MODE)
print("Kubeconfig path :", KUBECONFIG_PATH)

# Initialize API clients
core = client.CoreV1Api()
apps = client.AppsV1Api()
networking = client.NetworkingV1Api()
version_api = client.VersionApi()

version_info = version_api.get_code()
print("Kubernetes ver. :", version_info.git_version)

In [ ]:
NAMESPACE = "llm-observability"

# Fetch all resources
nodes = core.list_node().items
all_pods = core.list_pod_for_all_namespaces().items
all_services = core.list_service_for_all_namespaces().items
all_deployments = apps.list_deployment_for_all_namespaces().items
all_statefulsets = apps.list_stateful_set_for_all_namespaces().items

# Filter for your application namespace
app_pods = [pod for pod in all_pods if pod.metadata.namespace == NAMESPACE]
app_services = [svc for svc in all_services if svc.metadata.namespace == NAMESPACE]
app_deployments = [dep for dep in all_deployments if dep.metadata.namespace == NAMESPACE]
app_statefulsets = [st for st in all_statefulsets if st.metadata.namespace == NAMESPACE]

print(f"Found {len(nodes)} node(s)")
print(f"Found {len(app_pods)} pods in namespace {NAMESPACE}")
print(f"Found {len(app_services)} services in namespace {NAMESPACE}")

In [ ]:
def nx_to_cytoscape(G: nx.Graph, layout: str = "spring", **layout_kwargs) -> tuple[list, list]:
    """
    Convert a NetworkX graph to Cytoscape elements.
    Returns a list of elements (nodes + edges) and a layout dict for the preset positions.
    """
    # Compute positions
    if layout == "spring":
        pos = nx.spring_layout(G, **layout_kwargs)
    elif layout == "kamada_kawai":
        pos = nx.kamada_kawai_layout(G, **layout_kwargs)
    elif layout == "circular":
        pos = nx.circular_layout(G, **layout_kwargs)
    else:
        raise ValueError(f"Unsupported layout: {layout}")

    elements = []
    # Nodes
    for node, data in G.nodes(data=True):
        x, y = pos[node]
        # Scale to reasonable cytoscape coordinates (e.g., between -300 and 300)
        x = x * 500
        y = y * 500
        elements.append({
            'data': {
                'id': node,
                'label': data.get('label', node),
                'kind': data.get('kind', 'node'),
            },
            'position': {'x': x, 'y': y}
        })
    # Edges
    for u, v, data in G.edges(data=True):
        elements.append({
            'data': {
                'source': u,
                'target': v,
                'label': data.get('label', ''),
                'relation': data.get('relation', ''),
            }
        })
    return elements

In [ ]:
def build_generic_k3s_graph() -> nx.DiGraph:
    G = nx.DiGraph()
    # Add nodes
    G.add_node("k3s Server", kind="process", label="k3s Server")
    G.add_node("k3s Agent", kind="process", label="k3s Agent")
    G.add_node("API Server", kind="component")
    G.add_node("Scheduler", kind="component")
    G.add_node("Controller Manager", kind="component")
    G.add_node("Kine (etcd)", kind="component")
    G.add_node("kubelet", kind="component")
    G.add_node("containerd", kind="component")
    G.add_node("Flannel", kind="component")
    G.add_node("Tunnel Proxy", kind="component")
    for i in range(1, 4):
        G.add_node(f"Pod {i}", kind="pod", label=f"Pod {i}")

    # Add edges (relations)
    # Server -> components
    G.add_edge("k3s Server", "API Server", relation="contains")
    G.add_edge("k3s Server", "Scheduler", relation="contains")
    G.add_edge("k3s Server", "Controller Manager", relation="contains")
    G.add_edge("k3s Server", "Kine (etcd)", relation="contains")
    # Agent -> components
    G.add_edge("k3s Agent", "kubelet", relation="contains")
    G.add_edge("k3s Agent", "containerd", relation="contains")
    G.add_edge("k3s Agent", "Flannel", relation="contains")
    G.add_edge("k3s Agent", "Tunnel Proxy", relation="contains")
    # Dependencies between components
    G.add_edge("API Server", "Kine (etcd)", relation="stores data")
    G.add_edge("Scheduler", "API Server", relation="watches pods")
    G.add_edge("Controller Manager", "API Server", relation="watches resources")
    G.add_edge("kubelet", "API Server", relation="registers node")
    G.add_edge("kubelet", "containerd", relation="manages containers")
    G.add_edge("kubelet", "Flannel", relation="configures network")
    G.add_edge("Flannel", "Tunnel Proxy", relation="network plugin")
    # kubelet -> pods
    for i in range(1, 4):
        G.add_edge("kubelet", f"Pod {i}", relation="runs")
    return G


In [14]:
from jupyter_dash import JupyterDash
import dash_cytoscape as cyto
from dash import html, dcc

# Build the graph
G_generic = build_generic_k3s_graph()
elements_generic = nx_to_cytoscape(G_generic, layout="spring", seed=42, k=1.5)

# Define a stylesheet for node appearance based on "kind"
stylesheet = [
    {
        'selector': 'node',
        'style': {
            'label': 'data(label)',
            'text-valign': 'center',
            'text-halign': 'center',
            'background-color': '#2a9d8f',
            'border-width': 1,
            'border-color': '#264653',
            'width': '60px',
            'height': '60px',
        }
    },
    {
        'selector': 'node[kind = "process"]',
        'style': {
            'background-color': '#264653',
            'shape': 'rectangle',
            'width': '80px',
            'height': '40px',
        }
    },
    {
        'selector': 'node[kind = "pod"]',
        'style': {
            'background-color': '#e9c46a',
        }
    },
    {
        'selector': 'edge',
        'style': {
            'curve-style': 'bezier',
            'target-arrow-shape': 'triangle',
            'label': 'data(relation)',
            'font-size': '10px',
            'text-rotation': 'autorotate',
        }
    }
]

app_generic = JupyterDash(__name__)
app_generic.layout = html.Div([
    html.H3("Generic k3s Architecture", style={"textAlign": "center"}),
    cyto.Cytoscape(
        id='generic-k3s',
        elements=elements_generic,
        layout={'name': 'preset'},  # use positions from nx layout
        style={'width': '100%', 'height': '600px'},
        stylesheet=stylesheet,
    )
])

app_generic.run_server(mode='inline', port=9050)

AttributeError: 'super' object has no attribute 'run_server'